# License

Copyright 2026 Sonic Operations Ltd.
This file is part of the Daphne consensus development infrastructure for Sonic.

Daphne is free software: you can redistribute it and/or modify
it under the terms of the GNU Lesser General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

Daphne is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU Lesser General Public License for more details.

You should have received a copy of the GNU Lesser General Public License
along with Daphne. If not, see <http://www.gnu.org/licenses/>.

# Consensus Study Analysis

This notebook imports data collected by Daphne's consensus study, analyzes
scalability and performance metrics of consensus protocols, and visualizes
results.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../data.parquet' with the path to your data file.
# As a reminder, please run `go run ./daphne study consensus` to produce this 
# file. Data is loaded in chunks and processed to categorize columns to limit 
# peak memory usage.
data = pd.read_parquet(
    '../../data.parquet',
    columns=['timestamp', 'mark', 'hash', 'rid', 'Consensus', 'block', 'block_timestamp'],
)

for col in ['mark', 'Consensus']:
    data[col] = data[col].astype('category')

## 2. Time to Finality
The following chart compares the time-to-finality distribution of the various
consensus algorithms. It uses the following definition of time-to-finality:

> The time from submitting the transaction to the first time the transaction got
confirmed in a block by any node.

Future work may refine this definition to more meaningful definitions.

In [ ]:
# Filter only relevant events
tx_submitted = data[data['mark'] == 'TxSubmitted'][['timestamp', 'hash', 'rid']]
tx_finalized = data[data['mark'] == 'TxFinalized'][['timestamp', 'hash', 'rid']]

# Get first TxFinalized per transaction (by hash and rid)
tx_finalized_first = tx_finalized.sort_values('timestamp').drop_duplicates(['hash', 'rid'], keep='first')

# Merge submitted and finalized to compute TTF
ttf = pd.merge(
    tx_submitted,
    tx_finalized_first,
    on=['hash', 'rid'],
    suffixes=('_submitted', '_finalized')
)
ttf['TTF'] = (ttf['timestamp_finalized'] - ttf['timestamp_submitted']) / 1e9  # convert ns to seconds

# Add Consensus info by merging with StudyStarted rows
run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Consensus']]
ttf = pd.merge(ttf, run_info, on='rid', how='left')

# Drop missing Consensus (shouldn't happen, but just in case)
ttf = ttf.dropna(subset=['Consensus'])

# Plot violin plot
plt.figure(figsize=(10, 6))
plt.grid(axis='y', linestyle=':')
sns.violinplot(data=ttf, x='Consensus', y='TTF', inner='quartile')
plt.ylabel('Time to Finality (seconds)')
plt.xlabel('Consensus Algorithm')
plt.title('TTF Distribution by Consensus Algorithm')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
block_proposed = data[data['mark'] == 'BlockProposed'][['block_timestamp', 'block', 'rid']]
block_proposed = block_proposed.drop_duplicates(subset=['block', 'rid'])

run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Consensus']]
block_timestamps = pd.merge(block_proposed, run_info, on='rid', how='left')

# pair block timestamps with timestamps offseted by one to compute block times
block_timestamps_offset = block_timestamps.copy()
block_timestamps_offset['block'] = block_timestamps_offset['block'] - 1
block_times = pd.merge(block_timestamps, block_timestamps_offset, on=['Consensus', 'block', 'rid'], suffixes=('_curr', '_next'))
block_times['block_time'] = (block_times['block_timestamp_next'] - block_times['block_timestamp_curr']) / 1e9 

plt.figure(figsize=(14, 12), dpi=100)
plt.grid(axis='y', linestyle=':')
sns.violinplot(data=block_times, x='Consensus', y='block_time', inner='quartile')
plt.ylabel('Block Time (seconds)')
plt.xlabel('Consensus Algorithm')
plt.title('Block Time Distribution by Consensus Algorithm')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

## 3. Skipped Transactions
The following chart displays the ratio of skipped and finalized transactions per consensus algorithm.

In [ ]:
# Filter only relevant events
tx_skipped = data[data['mark'] == 'TxSkipped'][['rid', 'block','hash']]
tx_skipped = tx_skipped.drop_duplicates(subset=['rid', 'block', 'hash'])

tx_finalized = data[data['mark'] == 'TxFinalized'][['rid', 'block','hash']]
tx_finalized = tx_finalized.drop_duplicates(subset=['rid', 'block', 'hash'])

# Add Consensus info by merging with StudyStarted rows
run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Consensus']]
tx_skipped = pd.merge(tx_skipped, run_info, on='rid', how='left')
tx_finalized = pd.merge(tx_finalized, run_info, on='rid', how='left')

# Compute the average number of skipped/finalized transactions per Consensus per run
skipped_counts = tx_skipped.groupby(['rid', 'Consensus'], observed=True).size().reset_index(name='SkippedCount')
skipped_counts = skipped_counts.groupby('Consensus', observed=False).aggregate("mean").reset_index()[['Consensus', 'SkippedCount']]

finalized_counts = tx_finalized.groupby(['rid', 'Consensus'], observed=True).size().reset_index(name='FinalizedCount')
finalized_counts = finalized_counts.groupby('Consensus', observed=False).aggregate("mean").reset_index()[['Consensus', 'FinalizedCount']]

merged_counts = pd.merge(finalized_counts, skipped_counts, on='Consensus', how='outer')

# Plot skipped/finalized transactions
plt.figure(figsize=(10, 6))
plt.bar(merged_counts['Consensus'], merged_counts['FinalizedCount'], label='Finalized')
plt.bar(merged_counts['Consensus'], merged_counts['SkippedCount'], label='Skipped', bottom=merged_counts['FinalizedCount'])
plt.legend()
plt.ylabel('Mean Number of Transactions')
plt.xlabel('Consensus Algorithm')
plt.title('Mean Skipped vs Finalized Transactions by Consensus Algorithm')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()